# GRACE-FO `MeTp` 迭代计算（D → C / M → T）

这版 notebook 按你的最新要求实现：

- 路线为 **D → C**
- 其中 **D 对应 M 星（发射星）**，**C 对应 T 星（接收星）**
- 使用 `ltc_dc_tpm_tsm_only.xlsx` 中按 `gps_time` 对齐的 **`TpMr`** 列作为 \(\Delta t_{TpMr}\)
- 主迭代公式按你给出的图中公式实现
- **只对 \(\Delta t_{MeTp}\) 做迭代**
- 在每次迭代里：
  - 先更新
    \[
    r_r = r_T\bigl(t_r-\Delta t_{TpMr}\bigr)
    \]
    \[
    r_e = r_M\bigl(t_r-\Delta t_{MeTp}^{(n)}-\Delta t_{TpMr}\bigr)
    \]
  - 再用更新后的 \(r_r\)、\(r_e\) 重新计算 **TPM** 和 **TSM**
- 不再使用前一版里对 PM/SM 的解析近似展开
- `r_T` 和 `r_M` 的时间展开仍保持程序原来的二阶泰勒形式：
  \[
  r(t_r-\varepsilon)\approx r(t_r)-\dot r(t_r)\varepsilon+\tfrac12 \ddot r(t_r)\varepsilon^2
  \]

## 本 notebook 中采用的实现约定

1. 0 阶取
   \[
   \Delta t_{MeTp}^{(0)} = t_{inst}
   \]
   其中 `t_inst` 由
   \[
   r_T(t_r-\Delta t_{TpMr}),\quad r_M(t_r-\Delta t_{TpMr})
   \]
   构造。

2. 在每轮迭代中，**TPM/TSM 都放在循环里计算**，因为其对应的 \(r_r\)、\(r_e\) 会随着 \(\Delta t_{MeTp}^{(n)}\) 更新。

3. 对于 TSM 中的 \(\Delta t_{SR}\)，按你最新要求使用当前迭代的
   \[
   t_r^{(n)} = t_r - \Delta t_{TpMr},\qquad
   t_e^{(n)} = t_r - \Delta t_{MeTp}^{(n)} - \Delta t_{TpMr}
   \]
   并在代码里取
   \[
   \Delta t_{SR}^{(n)} = \frac{\lVert r_r^{(n)}-r_e^{(n)}\rVert}{c_0}
   \]

如果你后面想把这一步改回 `|t_r - t_e|` 而不是再除以 `c0`，只需要改一行。

- 输出结果新增 `MeTp_minus_delta_t_inst = MeTp - Δt_inst`，用于表示剩余光时效应。

In [15]:

CONFIG = {
    # 输入文件
    "c_file": r"GNI1B_2022-06-05_C_04.txt",      # C = T 星（接收星）
    "d_file": r"GNI1B_2022-06-05_D_04.txt",      # D = M 星（发射星）
    "tpmr_xlsx": r"ltc_dc_tpm_tsm_only.xlsx",    # 读取其中的 delta_t_TpMr 列

    # 地球常数（直接写在开头）
    "GM": 0.3986004415E+15,   # m^3 / s^2
    "Re": 0.6378136460E+07,   # m

    # 迭代参数
    "tol": 1e-18,
    "max_iter": 3,

    # 只算前若干条；None 表示全算
    "max_rows": None,

    # 是否显示进度条
    "show_progress": True,

    # 输出文件
    "out_xlsx": "ltc_ctod_metp_iterative.xlsx",
}
CONFIG


{'c_file': 'GNI1B_2022-06-05_C_04.txt',
 'd_file': 'GNI1B_2022-06-05_D_04.txt',
 'tpmr_xlsx': 'ltc_dc_tpm_tsm_only.xlsx',
 'GM': 398600441500000.0,
 'Re': 6378136.46,
 'tol': 1e-18,
 'max_iter': 3,
 'max_rows': None,
 'show_progress': True,
 'out_xlsx': 'ltc_ctod_metp_iterative.xlsx'}

In [16]:

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None


C0 = 299792458.0
OMEGA_E_DEFAULT = np.array([0.0, 0.0, 7.2921150e-5], dtype=float)


def progress_iter(iterable, total: int, desc: str, enable: bool = True):
    if enable and tqdm is not None:
        return tqdm(iterable, total=total, desc=desc)
    return iterable


def norm3(x: np.ndarray) -> float:
    return float(np.linalg.norm(np.asarray(x, dtype=float)))


def unit3(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    n = norm3(x)
    if n == 0.0:
        raise ValueError("零向量无法单位化。")
    return x / n


def second_order_taylor_position(r_tr: np.ndarray,
                                 v_tr: np.ndarray,
                                 a_tr: np.ndarray,
                                 delta_t: float) -> np.ndarray:
    """
    r(tr - dt) ≈ r(tr) - v(tr) * dt + 0.5 * a(tr) * dt^2
    """
    r_tr = np.asarray(r_tr, dtype=float)
    v_tr = np.asarray(v_tr, dtype=float)
    a_tr = np.asarray(a_tr, dtype=float)
    dt = float(delta_t)
    return r_tr - v_tr * dt + 0.5 * a_tr * dt**2


@dataclass
class ModelConstants:
    GM: float
    Re: float


@dataclass
class OneWayIterationResult:
    delta_t: float
    history: List[Dict[str, float]]


def read_gni1b_txt(path: str | Path) -> pd.DataFrame:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"未找到文件: {path}")

    lines = path.read_text(encoding="utf-8", errors="ignore").splitlines()
    try:
        end_idx = next(i for i, line in enumerate(lines) if line.strip() == "# End of YAML header")
    except StopIteration as exc:
        raise ValueError(f"文件 {path} 未找到 '# End of YAML header'。") from exc

    col_names = [
        "gps_time", "sat_id", "coord_ref",
        "xpos", "ypos", "zpos",
        "xpos_err", "ypos_err", "zpos_err",
        "xvel", "yvel", "zvel",
        "xvel_err", "yvel_err", "zvel_err",
        "qualflg",
    ]

    df = pd.read_csv(
        path,
        sep=r"\s+",
        skiprows=end_idx + 1,
        header=None,
        names=col_names,
        engine="python",
    )

    df = df[["gps_time", "coord_ref", "xpos", "ypos", "zpos", "xvel", "yvel", "zvel"]].copy()

    bad = df.loc[df["coord_ref"] != "I"]
    if not bad.empty:
        raise ValueError(f"文件 {path} 中存在非惯性系记录，coord_ref 应为 'I'。")

    return df.reset_index(drop=True)


def finite_difference_weights_first_derivative(x_nodes: np.ndarray,
                                             x0: float) -> np.ndarray:
    """
    基于局部节点解 Vandermonde 线性方程，构造一阶导数有限差分权重。
    该实现可同时适用于：
    - 内部点 5 点中心差分
    - 边界点 5 点单边差分
    - 样本不足时自动降阶
    - 非严格等步长的 gps_time
    """
    x_nodes = np.asarray(x_nodes, dtype=float)
    n = x_nodes.size
    if n < 2:
        raise ValueError("求一阶导数至少需要 2 个点。")

    dx = x_nodes - float(x0)
    A = np.vstack([dx**k for k in range(n)])
    b = np.zeros(n, dtype=float)
    b[1] = 1.0
    return np.linalg.solve(A, b)


def finite_difference_first_derivative(y: np.ndarray,
                                       t: np.ndarray,
                                       stencil: int = 5) -> np.ndarray:
    """
    用高阶有限差分计算 dy/dt：
    - 默认优先使用 5 点模板
    - 边界自动切换为单边模板
    - 点数不足时自动降阶
    """
    y = np.asarray(y, dtype=float)
    t = np.asarray(t, dtype=float)

    if y.ndim != 1 or t.ndim != 1 or y.size != t.size:
        raise ValueError("y 和 t 必须为等长一维数组。")

    n = y.size
    if n < 2:
        raise ValueError("有限差分至少需要 2 个样本点。")

    use_stencil = int(max(2, min(int(stencil), n)))
    half = use_stencil // 2
    dydt = np.empty(n, dtype=float)

    for i in range(n):
        if n <= use_stencil:
            start = 0
            end = n
        else:
            start = i - half
            end = start + use_stencil
            if start < 0:
                start = 0
                end = use_stencil
            if end > n:
                end = n
                start = n - use_stencil

        t_nodes = t[start:end]
        y_nodes = y[start:end]
        w = finite_difference_weights_first_derivative(t_nodes, t[i])
        dydt[i] = float(np.dot(w, y_nodes))

    return dydt


def estimate_acceleration_from_velocity(df: pd.DataFrame) -> pd.DataFrame:
    """
    按与前面程序相同的处理方式：
    直接对速度序列做高阶有限差分，得到二阶展开所需加速度。
    """
    out = df.copy()

    t = out["gps_time"].to_numpy(dtype=float)
    vx = out["xvel"].to_numpy(dtype=float)
    vy = out["yvel"].to_numpy(dtype=float)
    vz = out["zvel"].to_numpy(dtype=float)

    out["xacc"] = finite_difference_first_derivative(vx, t, stencil=5)
    out["yacc"] = finite_difference_first_derivative(vy, t, stencil=5)
    out["zacc"] = finite_difference_first_derivative(vz, t, stencil=5)
    return out


def prepare_satellite_dataframe(path: str | Path) -> pd.DataFrame:
    df = read_gni1b_txt(path)
    df = estimate_acceleration_from_velocity(df)
    return df


def read_tpmr_xlsx(path: str | Path) -> pd.DataFrame:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"未找到文件: {path}")

    df = pd.read_excel(path)
    need_cols = {"gps_time", "delta_t_TpMr"}
    if not need_cols.issubset(df.columns):
        raise ValueError(f"{path} 中必须包含列: {sorted(need_cols)}")
    out = df[["gps_time", "delta_t_TpMr"]].copy()
    out["gps_time"] = out["gps_time"].astype(float)
    out["delta_t_TpMr"] = out["delta_t_TpMr"].astype(float)
    return out


def compute_tpm(rr_gcrs: np.ndarray,
                re_gcrs: np.ndarray,
                GM: float,
                c0: float = C0) -> float:
    rrn = norm3(rr_gcrs)
    ren = norm3(re_gcrs)
    dr = norm3(rr_gcrs - re_gcrs)

    num = rrn + ren + dr
    den = rrn + ren - dr
    if den <= 0.0:
        raise ValueError("T_PM 计算失败：对数分母 <= 0。")

    return 2.0 * GM / c0**3 * np.log(num / den)


def compute_tsm(rr_gcrs: np.ndarray,
                re_gcrs: np.ndarray,
                d0_inst: np.ndarray,
                delta_t_sr: float,
                omega_e_vec: np.ndarray,
                GM: float,
                Re: float,
                c0: float = C0) -> float:
    re_n = norm3(re_gcrs)
    rr_n = norm3(rr_gcrs)
    cross_term = np.cross(omega_e_vec, re_gcrs)
    dot_term = float(np.dot(cross_term, d0_inst))
    geom_term = (1.0 / re_n**3) + (1.0 / rr_n**3)

    return (2.0 * GM * Re**2 / (5.0 * c0**3)) * dot_term * geom_term * delta_t_sr


In [17]:


def iterate_me_to_tp_delay(
    tr_seconds: float,
    rM_tr: np.ndarray,
    vM_tr: np.ndarray,
    aM_tr: np.ndarray,
    rT_tr: np.ndarray,
    vT_tr: np.ndarray,
    aT_tr: np.ndarray,
    delta_t_tpmr: float,
    constants: ModelConstants,
    omega_e_vec: np.ndarray = OMEGA_E_DEFAULT,
    tol: float = 1e-18,
    max_iter: int = 20,
) -> OneWayIterationResult:
    """
    按当前物理约定实现 MeTp：

        Δt_MeTp^(n+1)
          = | r_T(tr - Δt_TpMr) - r_M(tr - Δt_MeTp^(n) - Δt_TpMr) | / c0
            + T_PM^(n) + T_SM^(n)

    其中：
    - 发射星 = D 星（本程序中对应 M）
    - 接收星 = C 星（本程序中对应 T）
    - Δt_TpMr 来自外部 xlsx 的 delta_t_TpMr 列
    - 主迭代里的几何项对 r_T 和 r_M 都做展开
    - TPM / TSM 都放在循环内，因为 r_r 和 r_e 会更新
    - TSM 的 Δt_SR 使用当前迭代更新后的两点位置构造：
          Δt_SR^(n) = |r_r^(n) - r_e^(n)| / c0
    """
    rM_tr = np.asarray(rM_tr, dtype=float)
    vM_tr = np.asarray(vM_tr, dtype=float)
    aM_tr = np.asarray(aM_tr, dtype=float)

    rT_tr = np.asarray(rT_tr, dtype=float)
    vT_tr = np.asarray(vT_tr, dtype=float)
    aT_tr = np.asarray(aT_tr, dtype=float)

    # 0 阶：在 Tp 事件附近构造瞬时量
    rT_tp0 = second_order_taylor_position(rT_tr, vT_tr, aT_tr, 0)
    rM_tp0 = second_order_taylor_position(rM_tr, vM_tr, aM_tr, 0)

    rel_inst = rT_tp0 - rM_tp0
    delta_t_inst = norm3(rel_inst) / C0

    delta_t_n = delta_t_inst
    history: List[Dict[str, float]] = []

    for n in range(max_iter):
        # 当前迭代对应的接收/发射事件时刻
        tr_iter = tr_seconds - delta_t_tpmr
        te_iter = tr_seconds - delta_t_tpmr - delta_t_n

        # 当前迭代更新后的两点位置：r_r, r_e
        rr_n = second_order_taylor_position(rT_tr, vT_tr, aT_tr, delta_t_tpmr)
        re_n = second_order_taylor_position(rM_tr, vM_tr, aM_tr, delta_t_tpmr + delta_t_n)

        # 当前迭代 LOS 方向
        d0_n = unit3(rr_n - re_n)

        # 当前迭代的 PM / SM
        tpm_n = compute_tpm(rr_n, re_n, constants.GM, C0)

        delta_t_sr_n = norm3(rr_n - re_n) / C0
        tsm_n = compute_tsm(
            rr_gcrs=rr_n,
            re_gcrs=re_n,
            d0_inst=d0_n,
            delta_t_sr=delta_t_sr_n,
            omega_e_vec=omega_e_vec,
            GM=constants.GM,
            Re=constants.Re,
            c0=C0,
        )

        geom_n = norm3(rr_n - re_n) / C0
        delta_t_np1 = geom_n + tpm_n + tsm_n

        history.append({
            "iter": int(n),
            "delta_t_n": float(delta_t_n),
            "delta_t_np1": float(delta_t_np1),
            "abs_update": float(abs(delta_t_np1 - delta_t_n)),
            "delta_t_TpMr": float(delta_t_tpmr),
            "delta_t_inst": float(delta_t_inst),
            "tr_iter": float(tr_iter),
            "te_iter": float(te_iter),
            "delta_t_sr_n": float(delta_t_sr_n),
            "geom_n": float(geom_n),
            "T_PM_n": float(tpm_n),
            "T_SM_n": float(tsm_n),
        })

        if abs(delta_t_np1 - delta_t_n) < tol:
            return OneWayIterationResult(delta_t=float(delta_t_np1), history=history)

        delta_t_n = delta_t_np1

    return OneWayIterationResult(delta_t=float(delta_t_n), history=history)


def compute_d_to_c_metp(
    c_file: str | Path,
    d_file: str | Path,
    tpmr_xlsx: str | Path,
    constants: ModelConstants,
    tol: float = 1e-18,
    max_iter: int = 20,
    max_rows: Optional[int] = None,
    show_progress: bool = True,
) -> Tuple[pd.DataFrame, List[List[Dict[str, float]]]]:
    """
    仅计算 D 发射 -> C 接收（即 M -> T）路线的 MeTp。
    并从外部 xlsx 读取同一 gps_time 下的 TpMr。
    """
    m_df = prepare_satellite_dataframe(d_file)   # D = 发射星 = M
    t_df = prepare_satellite_dataframe(c_file)   # C = 接收星 = T
    tpmr_df = read_tpmr_xlsx(tpmr_xlsx)

    merged = (
        pd.merge(m_df, t_df, on="gps_time", suffixes=("_M", "_T"), how="inner")
        .merge(tpmr_df, on="gps_time", how="inner")
        .sort_values("gps_time")
        .reset_index(drop=True)
    )

    if max_rows is not None:
        merged = merged.iloc[:max_rows].copy()

    out_rows = []
    histories_all: List[List[Dict[str, float]]] = []

    iterator = progress_iter(
        merged.itertuples(index=False),
        total=len(merged),
        desc="D -> C (M -> T) MeTp 计算进度",
        enable=show_progress,
    )

    for row in iterator:
        tr = float(row.gps_time)

        # 发射星 D = M
        rM = np.array([row.xpos_M, row.ypos_M, row.zpos_M], dtype=float)
        vM = np.array([row.xvel_M, row.yvel_M, row.zvel_M], dtype=float)
        aM = np.array([row.xacc_M, row.yacc_M, row.zacc_M], dtype=float)

        # 接收星 C = T
        rT = np.array([row.xpos_T, row.ypos_T, row.zpos_T], dtype=float)
        vT = np.array([row.xvel_T, row.yvel_T, row.zvel_T], dtype=float)
        aT = np.array([row.xacc_T, row.yacc_T, row.zacc_T], dtype=float)

        result = iterate_me_to_tp_delay(
            tr_seconds=tr,
            rM_tr=rM,
            vM_tr=vM,
            aM_tr=aM,
            rT_tr=rT,
            vT_tr=vT,
            aT_tr=aT,
            delta_t_tpmr=float(row.delta_t_TpMr),
            constants=constants,
            omega_e_vec=OMEGA_E_DEFAULT,
            tol=tol,
            max_iter=max_iter,
        )

        last = result.history[-1]
        delta_t_inst = float(last["delta_t_inst"])
        out_rows.append({
            "gps_time": tr,
            "MeTp": result.delta_t,
            "delta_t_inst": delta_t_inst,
            "MeTp_minus_delta_t_inst": float(result.delta_t - delta_t_inst),
            "tr_iter": float(last["tr_iter"]),
            "te_iter": float(last["te_iter"]),
            "delta_t_sr": float(last["delta_t_sr_n"]),
            "geom": float(last["geom_n"]),
            "T_PM": float(last["T_PM_n"]),
            "T_SM": float(last["T_SM_n"]),
            "n_iter": len(result.history),
            "last_abs_update": float(last["abs_update"]),
        })
        histories_all.append(result.history)

    return pd.DataFrame(out_rows), histories_all


In [18]:

constants = ModelConstants(
    GM=float(CONFIG["GM"]),
    Re=float(CONFIG["Re"]),
)

out_df, histories = compute_d_to_c_metp(
    c_file=CONFIG["c_file"],
    d_file=CONFIG["d_file"],
    tpmr_xlsx=CONFIG["tpmr_xlsx"],
    constants=constants,
    tol=float(CONFIG["tol"]),
    max_iter=int(CONFIG["max_iter"]),
    max_rows=CONFIG["max_rows"],
    show_progress=bool(CONFIG["show_progress"]),
)

with pd.ExcelWriter(CONFIG["out_xlsx"], engine="openpyxl") as writer:
    out_df.to_excel(writer, sheet_name="MeTp", index=False)

print("计算完成。")
print(f"结果文件: {CONFIG['out_xlsx']}")
print("输出列：")
print(out_df.columns.tolist())

out_df.head()


D -> C (M -> T) MeTp 计算进度:   0%|          | 0/86400 [00:00<?, ?it/s]

计算完成。
结果文件: ltc_ctod_metp_iterative.xlsx
输出列：
['gps_time', 'MeTp', 'delta_t_inst', 'MeTp_minus_delta_t_inst', 'tr_iter', 'te_iter', 'delta_t_sr', 'geom', 'T_PM', 'T_SM', 'n_iter', 'last_abs_update']


,gps_time,MeTp,delta_t_inst,MeTp_minus_delta_t_inst,tr_iter,te_iter,delta_t_sr,geom,T_PM,T_SM,n_iter,last_abs_update
0,707659200.0,0.00065,0.00065,1.656224e-08,7.076592e+08,7.076592e+08,0.00065,0.00065,8.419424e-13,7.429761e-21,3,1.203464e-17
1,707659201.0,0.00065,0.00065,1.656224e-08,7.076592e+08,7.076592e+08,0.00065,0.00065,8.419436e-13,7.430786e-21,3,1.225148e-17
2,707659202.0,0.00065,0.00065,1.656224e-08,7.076592e+08,7.076592e+08,0.00065,0.00065,8.419448e-13,7.431811e-21,3,1.127570e-17
3,707659203.0,0.00065,0.00065,1.656224e-08,7.076592e+08,7.076592e+08,0.00065,0.00065,8.419461e-13,7.432836e-21,3,1.203464e-17
4,707659204.0,0.00065,0.00065,1.656224e-08,7.076592e+08,7.076592e+08,0.00065,0.00065,8.419473e-13,7.433861e-21,3,8.890458e-18
